# Детекция структуры и парсинг ячеек
Вход: одна страница после предобработки. Выход: список ячеек с координатами и вырезанными изображениями.

**Пайплайн:**
1. Бинаризация
2. Выделение горизонтальных и вертикальных линий
3. Построение маски сетки и поиск пересечений
4. Восстановление ячеек из пересечений
5. Визуализация и экспорт

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple
import json

# ─── Конфигурация ────────────────────────────────────────────────────────────
INPUT_PATH  = 'preproc_results/00000014_left.jpg'   # <- страница после предобработки
OUTPUT_DIR  = Path('cells')
OUTPUT_DIR.mkdir(exist_ok=True)

# Бинаризация
BINARIZE_BLOCK  = 17    # размер блока для адаптивного порога (нечётное число)
BINARIZE_C      = 10    # константа вычитания

# Морфология линий
H_LINE_SCALE    = 30    # делитель ширины для длины горизонтального ядра
V_LINE_SCALE    = 30    # делитель высоты для длины вертикального ядра
LINE_THICKNESS  = 4    # утолщение линий перед поиском пересечений

# Фильтрация ячеек
CELL_MIN_W      = 30    # минимальная ширина ячейки (пиксели)
CELL_MIN_H      = 20    # минимальная высота ячейки
CELL_MAX_W      = 2000  # максимальная ширина (отсекает всю страницу)
CELL_MAX_H      = 1000  # максимальная высота
CELL_PADDING    = 3     # отступ при вырезании ячейки

def show(images, titles, figsize=(18, 8), cmap='gray'):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1: axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap if img.ndim == 2 else None)
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.tight_layout(); plt.show()

print('Зависимости загружены ✓')

## Шаг 1 — Загрузка и бинаризация
Используем адаптивный порог (Sauvola-style через OpenCV `adaptiveThreshold`): он справляется
с неравномерным освещением лучше глобального порога.

In [ ]:
img_bgr  = cv2.imread(INPUT_PATH)
assert img_bgr is not None, f'Файл не найден: {INPUT_PATH}'

img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
h, w     = img_gray.shape
print(f'Размер страницы: {w}×{h} px')

# Адаптивная бинаризация: текст и линии → белые, фон → чёрный
binary = cv2.adaptiveThreshold(
    img_gray,
    maxValue=255,
    adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    thresholdType=cv2.THRESH_BINARY_INV,
    blockSize=BINARIZE_BLOCK,
    C=BINARIZE_C
)

show([img_rgb, binary], ['Исходная страница', 'Бинаризация (INV)'], figsize=(18, 10))

## Шаг 2 — Выделение линий таблицы
Морфологическая эрозия + дилатация:
- горизонтальное ядро длиной `width/H_LINE_SCALE` → оставляет только длинные горизонтальные линии
- вертикальное ядро → только вертикальные

Параметры `H_LINE_SCALE` / `V_LINE_SCALE` — ключевые: чем меньше значение, тем длиннее ядро
и тем строже фильтр (короткие штрихи не проходят).

In [ ]:
def extract_lines(binary: np.ndarray,
                  h_scale: int, v_scale: int,
                  thickness: int = 2) -> Tuple[np.ndarray, np.ndarray]:
    """Возвращает маски горизонтальных и вертикальных линий."""
    h, w = binary.shape

    # ── Горизонтальные ──────────────────────────────────────────────────────
    h_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT, (max(w // h_scale, 5), 1))
    h_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_kernel)
    h_lines = cv2.dilate(
        h_lines, cv2.getStructuringElement(cv2.MORPH_RECT, (1, thickness)))

    # ── Вертикальные ────────────────────────────────────────────────────────
    v_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT, (1, max(h // v_scale, 5)))
    v_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, v_kernel)
    v_lines = cv2.dilate(
        v_lines, cv2.getStructuringElement(cv2.MORPH_RECT, (thickness, 1)))

    return h_lines, v_lines


h_lines, v_lines = extract_lines(binary, H_LINE_SCALE, V_LINE_SCALE, LINE_THICKNESS)

# Цветная визуализация: красный = горизонт., синий = вертикаль.
vis_lines = np.zeros((h, w, 3), dtype=np.uint8)
vis_lines[h_lines > 0] = [220, 60, 60]
vis_lines[v_lines > 0] = [60, 100, 220]

show([h_lines, v_lines, vis_lines],
     ['Горизонтальные линии', 'Вертикальные линии', 'Оба слоя (кр/синий)'],
     figsize=(18, 7))

print(f'Горизонтальных пикселей: {h_lines.sum()//255:,}')
print(f'Вертикальных пикселей:   {v_lines.sum()//255:,}')

## Шаг 3 — Маска сетки и нахождение пересечений
Складываем два слоя линий → получаем маску всей сетки.
Пересечения (узлы сетки) — это AND горизонтальных и вертикальных масок.

In [ ]:
# Маска сетки
grid_mask = cv2.add(h_lines, v_lines)

# Пересечения — пиксели где есть И горизонталь, И вертикаль
intersections = cv2.bitwise_and(h_lines, v_lines)

# Небольшая дилатация пересечений чтобы объединить соседние точки
inter_dilated = cv2.dilate(
    intersections,
    cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
)

# Центроиды каждого узла
n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    inter_dilated, connectivity=8
)
# Пропускаем компонент 0 (фон)
nodes = centroids[1:].astype(int)   # shape (N, 2): [[x, y], ...]
print(f'Найдено узлов сетки: {len(nodes)}')

# Визуализация
vis_grid = cv2.cvtColor(grid_mask, cv2.COLOR_GRAY2RGB)
for x, y in nodes:
    cv2.circle(vis_grid, (x, y), 6, (255, 80, 0), -1)

show([grid_mask, vis_grid],
     ['Маска сетки', f'Узлы сетки ({len(nodes)} шт.)'],
     figsize=(18, 9))

## Шаг 4 — Восстановление ячеек
Два подхода работают в связке:
1. **Контурный** — `findContours` на маске сетки, отбираем прямоугольные контуры
2. **Проекционный** — если контурный не добрал ячейки, анализируем профили линий

Для чётких линий (наш случай) контурного метода обычно достаточно.

In [ ]:
@dataclass
class Cell:
    row: int
    col: int
    x: int
    y: int
    w: int
    h: int
    image: np.ndarray = field(repr=False, default=None)


def find_cells(grid_mask: np.ndarray,
               img: np.ndarray,
               min_w: int = 30, min_h: int = 20,
               max_w: int = 2000, max_h: int = 1000,
               padding: int = 3) -> List[Cell]:
    """
    Находит ячейки по контурам сетки, сортирует по строкам и столбцам,
    вырезает изображение каждой ячейки из оригинала.
    """
    # Инвертируем: ищем контуры ВНУТРИ ячеек (белые области)
    inv = cv2.bitwise_not(grid_mask)

    contours, _ = cv2.findContours(
        inv, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE
    )

    boxes = []
    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)
        if min_w <= bw <= max_w and min_h <= bh <= max_h:
            # Проверяем прямоугольность контура
            rect_area = bw * bh
            cnt_area  = cv2.contourArea(cnt)
            if cnt_area > 0 and (rect_area / cnt_area) < 2.5:
                boxes.append((x, y, bw, bh))

    if not boxes:
        return []

    # Убираем дубликаты (контуры с почти одинаковыми bbox)
    boxes = sorted(set(boxes))

    # Кластеризуем по строкам: группируем bbox с близким y (±10% высоты ячейки)
    row_tol = np.median([bh for _, _, _, bh in boxes]) * 0.4
    rows: List[List] = []
    for box in sorted(boxes, key=lambda b: b[1]):
        placed = False
        for row in rows:
            if abs(box[1] - row[0][1]) < row_tol:
                row.append(box)
                placed = True
                break
        if not placed:
            rows.append([box])

    cells = []
    ih, iw = img.shape[:2]
    for r_idx, row in enumerate(rows):
        for c_idx, (x, y, bw, bh) in enumerate(
                sorted(row, key=lambda b: b[0])):
            # Вырезаем с отступом, не выходя за границы
            x1 = max(0, x + padding)
            y1 = max(0, y + padding)
            x2 = min(iw, x + bw - padding)
            y2 = min(ih, y + bh - padding)
            crop = img[y1:y2, x1:x2]
            cells.append(Cell(r_idx, c_idx, x, y, bw, bh, crop))

    return cells


cells = find_cells(
    grid_mask, img_rgb,
    min_w=CELL_MIN_W, min_h=CELL_MIN_H,
    max_w=CELL_MAX_W, max_h=CELL_MAX_H,
    padding=CELL_PADDING
)

n_rows = max(c.row for c in cells) + 1 if cells else 0
n_cols = max(c.col for c in cells) + 1 if cells else 0
print(f'Найдено ячеек: {len(cells)}  ({n_rows} строк × {n_cols} столбцов)')

## Шаг 5 — Визуализация найденных ячеек

In [ ]:
# Рисуем bbox всех ячеек на оригинале
vis_cells = img_rgb.copy()

# Палитра по строкам
palette = plt.cm.tab20.colors
def row_color(r):
    c = palette[r % len(palette)]
    return tuple(int(x * 255) for x in c[:3])

for cell in cells:
    color = row_color(cell.row)
    cv2.rectangle(vis_cells,
                  (cell.x, cell.y),
                  (cell.x + cell.w, cell.y + cell.h),
                  color, 2)
    cv2.putText(vis_cells,
                f'r{cell.row}c{cell.col}',
                (cell.x + 4, cell.y + 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

fig, ax = plt.subplots(figsize=(16, 20))
ax.imshow(vis_cells)
ax.set_title(f'Найдено {len(cells)} ячеек — цвет = строка', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

## Шаг 6 — Просмотр вырезанных ячеек по строкам
Показываем все ячейки одной строки. Меняй `PREVIEW_ROW` для просмотра других строк.

In [ ]:
PREVIEW_ROW = 0   # <- номер строки таблицы для предпросмотра

row_cells = [c for c in cells if c.row == PREVIEW_ROW]
if not row_cells:
    print(f'Строка {PREVIEW_ROW} не найдена')
else:
    n = len(row_cells)
    fig, axes = plt.subplots(1, n, figsize=(min(n * 3, 20), 4))
    if n == 1: axes = [axes]
    for ax, cell in zip(axes, row_cells):
        ax.imshow(cell.image)
        ax.set_title(f'col {cell.col}\n{cell.w}×{cell.h}', fontsize=9)
        ax.axis('off')
    plt.suptitle(f'Строка {PREVIEW_ROW} — {n} ячеек', fontsize=12)
    plt.tight_layout()
    plt.show()

## Шаг 7 — Диагностика: гистограмма размеров ячеек
Помогает подобрать `CELL_MIN_W/H` и выявить аномалии (склеенные или слишком маленькие ячейки).

In [ ]:
widths  = [c.w for c in cells]
heights = [c.h for c in cells]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.hist(widths,  bins=40, color='steelblue', edgecolor='white')
ax1.set_title('Распределение ширин ячеек')
ax1.set_xlabel('Ширина (пикс.)')
ax1.axvline(np.median(widths),  color='red', linestyle='--',
            label=f'Медиана: {np.median(widths):.0f}')
ax1.legend()

ax2.hist(heights, bins=40, color='darkorange', edgecolor='white')
ax2.set_title('Распределение высот ячеек')
ax2.set_xlabel('Высота (пикс.)')
ax2.axvline(np.median(heights), color='red', linestyle='--',
            label=f'Медиана: {np.median(heights):.0f}')
ax2.legend()

plt.tight_layout()
plt.show()

print(f'Ширина:  min={min(widths)}, max={max(widths)}, median={np.median(widths):.0f}')
print(f'Высота:  min={min(heights)}, max={max(heights)}, median={np.median(heights):.0f}')

## Шаг 8 — Сохранение результатов
Сохраняем каждую ячейку как отдельный файл и метаданные в JSON.

In [ ]:
stem = Path(INPUT_PATH).stem
meta = []

for cell in cells:
    fname = f'{stem}_r{cell.row:02d}_c{cell.col:02d}.jpg'
    fpath = OUTPUT_DIR / fname
    if cell.image is not None and cell.image.size > 0:
        cv2.imwrite(str(fpath),
                    cv2.cvtColor(cell.image, cv2.COLOR_RGB2BGR))
    meta.append({
        'file':   fname,
        'row':    cell.row,
        'col':    cell.col,
        'x':      cell.x,
        'y':      cell.y,
        'width':  cell.w,
        'height': cell.h,
    })

meta_path = OUTPUT_DIR / f'{stem}_cells.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump({'source': INPUT_PATH,
               'total_cells': len(cells),
               'cells': meta}, f, ensure_ascii=False, indent=2)

print(f'Сохранено {len(cells)} ячеек → {OUTPUT_DIR.resolve()}')
print(f'Метаданные → {meta_path}')